# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook gọi `run_lab.py`, nên cùng một mã chạy được trên Colab hoặc VM.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Colab: khai báo Secrets KAGGLE_USERNAME/KAGGLE_KEY. VM: export hai biến trước khi mở notebook.
import os
try:
    from google.colab import userdata
    for key in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
except ImportError:
    pass
assert os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'), 'export Kaggle credentials first'

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from scripts.labcommon import read_jsonl
train_rows = read_jsonl('data/splits/train.jsonl')
print('train images:', len(train_rows), 'first:', train_rows[0])
assert len({r['session'] for r in train_rows}) > 1

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from run_lab import yolo_to_box, objects_to_answer
print(objects_to_answer(train_rows[0]['objects']))
assert yolo_to_box('pothole', [0.5, 0.5, 0.2, 0.2])['bbox_2d'] == [400, 400, 600, 600]

## 3. E0: zero-shot

In [ ]:
!python run_lab.py --experiment e0 --data data --output work/e0.jsonl

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from run_lab import encode_example, collate
print('Training encoding is implemented in run_lab.py; it masks all prompt/image tokens.')

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from run_lab import apply_setup
print('LoRA setup is applied inside the E1/E2/E3 runner commands.')

## 6. Vòng huấn luyện

In [ ]:
!python run_lab.py --experiment e1 --data data --output work/e1.jsonl --run-dir work/e1

## 7. Suy luận và chấm E1–E3

In [ ]:
!python run_lab.py --experiment e2 --data data --output work/e2.jsonl --run-dir work/e2
!python run_lab.py --experiment e3 --data data --output work/e3.jsonl --run-dir work/e3

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
!python run_lab.py --experiment e4 --rank 4 --data data --output work/e4_rank_4.jsonl --run-dir work/e4_rank_4
!python run_lab.py --experiment e4 --rank 64 --data data --output work/e4_rank_64.jsonl --run-dir work/e4_rank_64
# E5 GRPO là phần bonus; reward IoU có thể kiểm thử độc lập trước khi thêm TRL trainer.